# 01 · Scrape The Current's playlist history

Grabs every play from **2005-12-22** (first broadcast) through yesterday from
`https://www.thecurrent.org/playlist/the-current/YYYY-MM-DD`. One request = one full day.

- **Resumable:** each day is saved to `data/raw/YYYY/YYYY-MM-DD.csv`, and days already saved are skipped. Stop at any time and rerun.
- **Polite:** 5s (+0–1s random) between requests. It stops by itself on a 403 or 3 failed days in a row.
- **Full run ≈ 12 hours.** For the full run, the terminal is more reliable than this notebook:
  `caffeinate -i .venv/bin/python scraper.py` (see README). Use this notebook for small ranges and to inspect the data.

In [ ]:
from datetime import date, timedelta
from scraper import scrape_range, retry_failed, combine, START_DATE

## Config

In [ ]:
START = START_DATE                          # date(2005, 12, 22)
END = date.today() - timedelta(days=1)     # yesterday
DELAY = 5                                  # seconds between requests

## Scrape
Interrupt the kernel (■) any time; rerunning this cell picks up where it left off.

In [ ]:
scrape_range(START, END, delay=DELAY)

## Retry any failed days

In [ ]:
retry_failed(delay=DELAY)

## Combine into `data/plays.parquet`

In [ ]:
df = combine()
df.head()

## Sanity checks

In [ ]:
df.info()

In [ ]:
# plays per year (2005 is only 10 days; the current year is partial)
df.groupby(df.played_at.dt.year).size()

In [ ]:
# share of rows with each field filled in
df.notna().mean().round(3)

In [ ]:
# days with suspiciously few plays (possible gaps in the station's data)
per_day = df.groupby(df.played_at.dt.date).size()
per_day[per_day < 150].sort_values()

## What's in the data?

Everything below reads the committed yearly files in `data/plays/` (kept current by the monthly GitHub Action)
through `plays.load_plays()`. That function also adds a few helper columns the analysis notebooks share.

In [ ]:
from plays import load_plays

plays = load_plays()
print(f"{len(plays):,} plays, {plays.played_at.min():%b %d, %Y} → {plays.played_at.max():%b %d, %Y}")
plays.sample(5, random_state=1)

## What does each column mean?

| column | what it is | notes |
|---|---|---|
| `played_at` | When the song aired | Central time (`America/Chicago`), to the second |
| `title` | Song title, as logged | Typed by DJs/automation, so capitalization and spelling vary. It also includes things that aren't songs (see below) |
| `artist` | Artist name, as logged | Collaborations (`X feat. Y`, `X & Y`) are their own artist names |
| `album` | Album name | About 99% filled. Some values are station-specific: **`89.3 In-Studio Performance`** marks recordings of in-studio sessions, and `Live Current Vol. N` / `Local Current Vol. N` are the station's own compilations |
| `art_url` | Album art image | Hosted at `albumart.publicradio.org`. Coverage is lower in early years (see below) |
| `play_id` | Unique ID for this airing | One row per `play_id`, so it's the dedupe key |
| `song_id` | The Current's ID for a track | The **same song can have several `song_id`s** (single vs. album vs. compilation version). To count songs, use `song_key` (below) |
| `duration` | Track length, `m:ss` | About 30,000 plays (mostly 2006–07) are logged as `0:00`, which means unknown. Shows and concerts are logged as one long entry |
| `album_mbid` | MusicBrainz release ID | For later lookups of release year, label or genre |
| `spotify_id` | Spotify track ID | Only about 30% filled |
| `artist_ulid` | The Current's artist ID | Almost 1:1 with `artist` |
| `album_ulid` | The Current's album ID | Always empty |
| `song_type` | Track type | Always `PopularSong` |
| `additional_text` | Extra notes | Always empty |
| `service_id` | Which stream | Always `1` (The Current's main stream) |

**Helper columns added by `load_plays()`** (not in the CSVs):

| column | what it is |
|---|---|
| `seconds` | `duration` in seconds. `NaN` when missing or `0:00` |
| `artist_key` | The artist name lowercased with accents stripped (`Poliça` → `polica`), so spelling variants count once |
| `song_key` | `title` and `artist`, normalized the same way, joined as `title\|artist`. This counts one song once across albums and compilations |
| `is_music` | `False` for entries that look like shows, segments, interviews, clips or full concerts |

In [ ]:
# Share of plays with each field filled in, by year
fill = plays.groupby(plays.played_at.dt.year)[["album", "art_url", "seconds", "album_mbid", "spotify_id"]].agg(
    lambda s: s.notna().mean()
)
fill.style.format("{:.0%}").background_gradient(cmap="Blues", vmin=0, vmax=1)

## What isn't a song?

The station's log includes more than songs:

- **Syndicated shows** are logged in segments, e.g. "Hour 2 Segment 1" by "American Routes"
- **Full concerts and specials** are logged as one 60–90 minute entry, e.g. "Live at First Avenue"
- **Full in-studio sessions** are logged as one entry, e.g. "In-Studio Performance Recorded on May 2, 2017"
- **Interviews, promos and short sound clips**, e.g. movie quotes played as bumpers

`is_music` flags these with a few title/artist patterns, plus a rule that anything over 20 minutes is a program, not a song.
The patterns are defined at the top of `plays.py`. The filter is approximate, but it catches the large, repeated cases.

In [ ]:
print(f"{(~plays.is_music).sum():,} of {len(plays):,} entries ({(~plays.is_music).mean():.2%}) flagged as not a song")

# Most common flagged entries
plays[~plays.is_music].groupby(["title", "artist"]).size().sort_values(ascending=False).head(15)